### Step 1: Import python libraries and api key

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI
import gradio as gr
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")

c:\Users\Sam\Documents\ai-engineering\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Set up Pushover

In [2]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

In [3]:
# test Pushover
import requests

def send_notification(message: str):
    payload={"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [12]:
#send_notification("Hello to myself from Pushover.")

### Step 3: Describe Pushover as an LLM tool

In [4]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send a push notification to the user's phone via Pushover. User this to alert the user of time-sensitive issues.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pusherover to the list of tools for the LLM

In [5]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 2b &3b & 4b: create new functions

In [8]:
import random

# simulate rolling a single six-sided die
def dice_roll():
    result=random.randint(1, 6)
    return result

# describe function for the LLM
roll_dice_function={
    "name": "dice_roll",
    "description": "Simulate rolling a single six-sided die and returns the results. User this when the user wants to roll a dice.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

# add function to list of tools of LLM
tools.append({"type": "function", "function": roll_dice_function})

### Step 5: Call the tool from an LLM

In [9]:
def handle_tool_call(tool_calls):
    tool_results=[]

    for tool_call in tool_calls:
        function_name=tool_call.function.name
        args=json.loads(tool_call.function.arguments) 
        #print(f"Calling function {function_name})  # for future debugging

        # Route to the appropriate function based on function_name
        if function_name=="send_notification":
        # Actually send the notification for the tool call
            send_notification(args["message"])
            content=f"Notification sent: {args['message']}"
        elif function_name=="dice_roll":
            content=f"Rolled: {dice_roll()}"
        #elif function_name=="insert_function_name_3"
        #content=insert_function_name_3(args["message"])
        else: 
            content=f"Unknown function: {function_name}"

        tool_call_result={
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }

        tool_results.append(tool_call_result)

    return tool_results

In [ ]:
client=OpenAI()
messages=[
    {"role": "user", "content": "Please do two things: \
     1) I'd like to roll 4 dices, and \
     2) Send me a notification with the highest of the rolls."
    }
    ]

response=client.chat.completions.create(
    model="gpt-4o-mini", 
    messages=messages, 
    tools=tools,
)

# Check if the model wants to call a tool
message = response.choices[0].message

# check if model wants to call a tool
while message.tool_calls:
    from pprint import pprint
    pprint(message.tool_calls)
    tool_results=handle_tool_call(message.tool_calls) # whole list of tool calls on purpose
    messages.append(message)
    messages.extend(tool_results)  # changed from append() to extend() when we switched to multiple tool handling
 
    response=client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools #will add this in the future
    )
    
    message=response.choices[0].message

    # Maybe consider ading protection from infinite consecutive tool calling solution
print(message.content)

[ChatCompletionMessageFunctionToolCall(id='call_g2E0RxsnwQeC1r9VwHYSeIhB', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_XlE2l74TKYFVeBiAMtAkUHKO', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_j4abekVMvgunkU5mPpsLyY2i', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_Ek6hCJGZF3aWNFtB3AbMER5F', function=Function(arguments='{}', name='dice_roll'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_6kC4dMtoT0E34CBCggUIgoSo', function=Function(arguments='{"message":"The highest roll among your 4 dices is 3."}', name='send_notification'), type='function')]
I rolled four dice for you, and the results were 2, 3, 3, and 3. I have sent you a notification with the highest roll, which is 3.


: 

In [30]:
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_stoWNOSho9TVUkVoGq1wG5dO', function=Function(arguments='{"message":"Congratulations on your amazing progress in the AI Engineering class! Your dedication and hard work are truly paying off. Keep pushing forward!"}', name='send_notification'), type='function')])


In [32]:
if message.tool_calls:
    tool_call=message.tool_calls[0]
    import json
    args=json.loads(tool_call.function.arguments)

    # Actually send the notification
    send_notification(args["message"])
    print(f"Send notification: {args["message"]}")

else: 
    print(message.content)



Send notification: Congratulations on your amazing progress in the AI Engineering class! Your dedication and hard work are truly paying off. Keep pushing forward!
